# Replikasi Terbatas — Hidayatullah, Apong, Lai & Qazi (2025)
## *Pre-trained language model for code-mixed text in Indonesian, Javanese, and English using transformer*

**Jurnal:** Social Network Analysis and Mining 15:30 · DOI `10.1007/s13278-025-01444-9`

---

## Hasil audit: modelnya publik, datanya tertutup

| Komponen | Status |
|---|---|
| Model IndoJavE (4 varian) | ✅ **publik di HuggingFace** |
| Korpus pre-training | ❌ *"cannot be publicly released because of data usage policy"* |
| Dataset fine-tuning (IDEN-SA, IDEN-LID) | ❌ *"available upon request"* |
| Kode | ❌ tidak ada repo |
| Pre-training from scratch | ❌ butuh ratusan GPU-jam |

### Apa yang bisa dan tidak bisa

**Tidak bisa direplikasi:** pembuatan model dari nol. Paper melatih BERT, RoBERTa, IndoBERTweet, dan IndoBERT dari awal di atas korpus code-mixed yang tidak akan pernah kamu dapatkan. Tanpa korpus itu, pre-training tidak mungkin diulang.

**Bisa dikerjakan:** memakai model IndoJavE yang sudah dirilis, lalu fine-tune untuk tugas sentimen — itulah yang dilakukan notebook ini. Tapi karena dataset IDEN-SA tertutup, angkanya **tidak akan sebanding** dengan Table 12 paper.

### Model yang terverifikasi ada di HuggingFace

| Nama di paper | Repo HF |
|---|---|
| IndoJavE-BERT | `fathan/indojave-codemixed-bert-base` |
| IndoJavE-RoBERTa | `fathan/indojave-codemixed-roberta-base` |
| IndoJavE-IndoBERTweet | `fathan/indojave-codemixed-indobertweet-base` |
| IndoJavE-IndoBERT | `fathan/indojave-codemixed-indobert-base` |

Plus model hasil fine-tune: `fathan/ijelid-ft-indojave-indobertweet` (LID), `fathan/indo_sentiment_indobertweet`, `fathan/emotcls_indobertweet`.

> Catatan: paper **tidak mencantumkan link HuggingFace sama sekali**. Repo di atas saya temukan lewat pencarian HF berdasarkan nama penulis pertama, bukan dari paper.


## 1. Setup

In [2]:
!pip install -q "torch>=2.1" "transformers>=4.40" pandas scikit-learn requests

import os, json, random, urllib.request
import numpy as np, pandas as pd
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.optim import AdamW

SEED = 42          # [ASUMSI] paper tidak menyebut seed
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def show(d):
    try:
        from IPython.display import display as _d; _d(d)
    except Exception:
        print(d.to_string())
print('device:', DEVICE)

device: cuda


## 2. Verifikasi Ketersediaan Model (tanpa unduh bobot)

Cek ke HF API dulu — kalau nama repo salah, ketahuan sebelum mengunduh ratusan MB.

In [3]:
MODELS = {
    'IndoJavE-BERT':        'fathan/indojave-codemixed-bert-base',
    'IndoJavE-RoBERTa':     'fathan/indojave-codemixed-roberta-base',
    'IndoJavE-IndoBERTweet':'fathan/indojave-codemixed-indobertweet-base',
    'IndoJavE-IndoBERT':    'fathan/indojave-codemixed-indobert-base',
}

def hf_exists(repo_id):
    url = f'https://huggingface.co/api/models/{repo_id}'
    try:
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req, timeout=30) as r:
            return r.status == 200
    except Exception:
        return False

status = {k: hf_exists(v) for k, v in MODELS.items()}
show(pd.DataFrame([{'Model': k, 'Repo': v, 'Ada': status[k]} for k, v in MODELS.items()]))

hilang = [k for k, ok in status.items() if not ok]
assert not hilang, f'repo tidak ditemukan: {hilang}'
print('[OK] keempat repo IndoJavE dapat diakses')

,Model,Repo,Ada
0,IndoJavE-BERT,fathan/indojave-codemixed-bert-base,True
1,IndoJavE-RoBERTa,fathan/indojave-codemixed-roberta-base,True
2,IndoJavE-IndoBERTweet,fathan/indojave-codemixed-indobertweet-base,True
3,IndoJavE-IndoBERT,fathan/indojave-codemixed-indobert-base,True


[OK] keempat repo IndoJavE dapat diakses


## 3. Kenapa Pre-training Tidak Bisa Direplikasi

Paper melatih dari nol. Perkiraan biaya (paper tidak melaporkan jam GPU):

- Korpus IndoJavE ≈ ratusan juta token teks code-mixed
- BERT-base pre-training standar: ~4 hari di 16 TPU, atau setara ~100+ jam GPU A100
- Tanpa korpus (tertutup), angka ini bahkan tidak bisa dimulai

Jadi notebook ini **tidak** mencoba pre-training. Yang dilakukan: memakai bobot rilis + fine-tune.

In [4]:
BIAYA = [
    ('Korpus pre-training', 'TERTUTUP', 'tidak bisa dimulai'),
    ('Pre-training from scratch', '~100+ jam GPU A100', 'tidak realistis di Colab'),
    ('Fine-tune model rilis', '~10-20 menit di T4', 'bisa'),
]
for nama, biaya, status in BIAYA:
    print(f'{nama:32s} {biaya:24s} {status}')

Korpus pre-training              TERTUTUP                 tidak bisa dimulai
Pre-training from scratch        ~100+ jam GPU A100       tidak realistis di Colab
Fine-tune model rilis            ~10-20 menit di T4       bisa


## 4. Sanity Check Model pada Kalimat Paper

Paper Table 9 memuat contoh prediksi LID (language identification) beserta token-tokennya. Kalimat aslinya bisa dipakai sebagai uji cepat: apakah model rilis menandai token sesuai paper?

`[ASUMSI]` Model LID yang dipakai = `fathan/ijelid-ft-indojave-indobertweet` (satu-satunya model LID IndoJavE yang terlihat publik). Paper tidak menyebut repo-nya.

In [5]:
SAMPLES = [
    "Kalo menurutku ya, pelayanane wis apik kok.",
    "Guys, sorry iki aku belum bisa join online meetingnya",
    "Udah capek2 bikin, eh malah gak kepake",
    "Ngerasain atmosphere pertandingan hari ini, amazing banget!",
    "Pancen wong iki, bales replynya lamaaa bgt!",
]

LID_REPO = 'fathan/ijelid-ft-indojave-indobertweet'

if hf_exists(LID_REPO):
    tok = AutoTokenizer.from_pretrained(LID_REPO)
    mdl = AutoModelForSequenceClassification.from_pretrained(LID_REPO).to(DEVICE).eval()
    id2lab = {i: l for i, l in enumerate(mdl.config.id2label.values())}
    print('label model:', id2lab)

    for s in SAMPLES:
        enc = tok(s, return_tensors='pt', truncation=True, max_length=64).to(DEVICE)
        with torch.no_grad():
            p = mdl(**enc).logits.argmax(-1)[0].item()
        print(f'  {s[:55]:57s} -> {id2lab.get(p, p)}')
else:
    print('model LID tidak tersedia — sanity check dilewati')

config.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/588 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/235k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/738k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

...

[transformers] BertForSequenceClassification LOAD REPORT from: fathan/ijelid-ft-indojave-indobertweet
Key                      | Status  | 
-------------------------+---------+-
bert.pooler.dense.bias   | MISSING | 
bert.pooler.dense.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


label model: {0: 'EN', 1: 'ID', 2: 'JV', 3: 'MIX_ID_EN', 4: 'MIX_ID_JV', 5: 'MIX_JV_EN', 6: 'OTH'}


model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

  Kalo menurutku ya, pelayanane wis apik kok.               -> JV
  Guys, sorry iki aku belum bisa join online meetingnya     -> EN
  Udah capek2 bikin, eh malah gak kepake                    -> JV
  Ngerasain atmosphere pertandingan hari ini, amazing ban   -> MIX_ID_EN
  Pancen wong iki, bales replynya lamaaa bgt!               -> JV


## 5. Template Fine-tune Sentimen (pakai CSV sendiri)

Dataset IDEN-SA tertutup, jadi sel ini **tidak bisa dijalankan tanpa data**. Sediakan CSV dengan kolom `text` dan `label` (mis. `positive`/`negative`/`neutral`).

Paper Table 12 melaporkan F1 IDEN-SA: IndoJavE-IndoBERTweet **81,90%**, IndoBERTweet 81,64%, IndoJavE-IndoBERT 78,44%, IndoJavE-RoBERTa 77,60%, IndoJavE-BERT 76,43%.

In [6]:
USE_UPLOAD = False
CSV_PATH = 'iden_sa.csv'      # kolom: text, label

if USE_UPLOAD and os.path.exists(CSV_PATH):
    df = pd.read_csv(CSV_PATH)
    LABELS = sorted(df['label'].unique())
    L2I = {l: i for i, l in enumerate(LABELS)}
    print(f'{len(df)} baris | {len(LABELS)} kelas: {LABELS}')

    REPO = MODELS['IndoJavE-IndoBERTweet']     # model terbaik di paper
    MAXLEN, BS, EPOCHS, LR = 64, 32, 3, 2e-5   # [ASUMSI] paper tidak merinci

    tok = AutoTokenizer.from_pretrained(REPO)

    class SA_DS(Dataset):
        def __init__(self, texts, labels):
            self.enc = tok(list(texts), truncation=True, padding='max_length',
                           max_length=MAXLEN, return_tensors='pt')
            self.y = torch.tensor([L2I[l] for l in labels])
        def __len__(self): return len(self.y)
        def __getitem__(self, i):
            return {'input_ids': self.enc['input_ids'][i],
                    'attention_mask': self.enc['attention_mask'][i],
                    'labels': self.y[i]}

    from sklearn.model_selection import train_test_split
    from sklearn.metrics import f1_score
    Xtr, Xte, ytr, yte = train_test_split(df['text'].tolist(), df['label'].tolist(),
                                          test_size=0.2, random_state=SEED,
                                          stratify=df['label'])
    dl_tr = DataLoader(SA_DS(Xtr, ytr), batch_size=BS, shuffle=True)
    dl_te = DataLoader(SA_DS(Xte, yte), batch_size=64)

    model = AutoModelForSequenceClassification.from_pretrained(
        REPO, num_labels=len(LABELS)).to(DEVICE)
    opt = AdamW(model.parameters(), lr=LR)

    for ep in range(EPOCHS):
        model.train(); tot = 0.0
        for b in dl_tr:
            b = {k: v.to(DEVICE) for k, v in b.items()}
            opt.zero_grad(); loss = model(**b).loss
            loss.backward(); opt.step(); tot += loss.item()
        print(f'epoch {ep+1}/{EPOCHS} loss={tot/len(dl_tr):.4f}')

    model.eval(); P, G = [], []
    with torch.no_grad():
        for b in dl_te:
            b = {k: v.to(DEVICE) for k, v in b.items()}
            P += model(**b).logits.argmax(-1).cpu().tolist(); G += b['labels'].cpu().tolist()
    print(f'\nF1 (weighted) = {f1_score(G, P, average="weighted"):.4f}')
else:
    print('USE_UPLOAD=False — isi CSV IDEN-SA dulu, atau minta ke penulis korespondensi.')
    print('Sel ini hanya template: strukturnya benar, tapi tidak bisa jalan tanpa data.')

USE_UPLOAD=False — isi CSV IDEN-SA dulu, atau minta ke penulis korespondensi.
Sel ini hanya template: strukturnya benar, tapi tidak bisa jalan tanpa data.


## 6. Tabel Perbandingan

In [7]:
PAPER_T12 = {   # Table 12, dataset IDEN-SA (sentimen ID-EN)
    'IndoJavE-IndoBERTweet': 81.90,
    'IndoJavE-IndoBERT':    78.44,
    'IndoJavE-RoBERTa':     77.60,
    'IndoJavE-BERT':        76.43,
    'IndoBERTweet':         81.64,
    'IndoBERT':             55.71,
    'mBERT':                66.52,
    'XLM-RoBERTa':          78.68,
}
rows = [{'Model': k, 'F1 paper (%)': v, 'F1 replikasi': 'butuh data IDEN-SA'}
        for k, v in sorted(PAPER_T12.items(), key=lambda x: -x[1])]
show(pd.DataFrame(rows))
print('\nTanpa dataset IDEN-SA, kolom replikasi tidak bisa diisi.')
print('Jangan isi dengan angka karangan.')

,Model,F1 paper (%),F1 replikasi
0,IndoJavE-IndoBERTweet,81.90,butuh data IDEN-SA
1,IndoBERTweet,81.64,butuh data IDEN-SA
2,XLM-RoBERTa,78.68,butuh data IDEN-SA
3,IndoJavE-IndoBERT,78.44,butuh data IDEN-SA
4,IndoJavE-RoBERTa,77.60,butuh data IDEN-SA
5,IndoJavE-BERT,76.43,butuh data IDEN-SA
6,mBERT,66.52,butuh data IDEN-SA
7,IndoBERT,55.71,butuh data IDEN-SA



Tanpa dataset IDEN-SA, kolom replikasi tidak bisa diisi.
Jangan isi dengan angka karangan.


## 7. Laporan Akhir

### Ringkasan paper
Memperkenalkan IndoJavE: 4 PLM untuk teks code-mixed Indonesia–Jawa–Inggris (BERT, RoBERTa, IndoBERTweet, IndoBERT), dilatih dengan dua pendekatan (from scratch vs transfer learning) di atas korpus tweet code-mixed. Diuji pada tugas downstream: LID (IDEN-LID) dan sentimen (IDEN-SA). Klaim: transfer learning lebih efisien; IndoJavE-IndoBERTweet terbaik (F1 94,95% untuk LID; 81,90% untuk sentimen).

### Sudah teruji vs belum

| | |
|---|---|
| **Sudah dijalankan & diverifikasi** | Keempat repo model IndoJavE ada dan dapat diakses di HuggingFace |
| **Belum diuji** | Load bobot + inferensi LID, fine-tune sentimen (butuh data IDEN-SA + GPU) |

### [ASUMSI]
1. `SEED = 42` — paper tidak menyebut
2. Model LID = `fathan/ijelid-ft-indojave-indobertweet` — paper tidak menyebut repo
3. Fine-tune: MAXLEN 64, batch 32, 3 epoch, lr 2e-5 — paper tidak merinci
4. Split 80:20 — paper tidak menyebut
5. Repo HF ditemukan lewat pencarian nama penulis, bukan dari paper

### Ancaman validitas
- **Korpus pre-training tertutup permanen** (kebijakan data X/Twitter). Klaim inti paper — "transfer learning lebih baik dari from scratch" — tidak bisa diverifikasi independen
- **Dataset fine-tuning tertutup** → seluruh Table 11, 12, dan angka F1 tidak bisa diuji
- Paper tidak mencantumkan link model maupun kode. Repo yang saya temukan belum tentu yang dipakai di paper (bisa versi berbeda)
- Perbandingan dengan baseline (IndoBERT 55,71%) sangat rendah untuk IndoBERT — indikasi baseline tidak di-tuning setara. Ini melemahkan klaim "IndoJavE mengalahkan model monolingual"
